# Notebook 05: Handling Class Imbalance

## 📚 Historical Context

**Timeline**: The Class Imbalance Problem (1990s-2024)

**The Discovery (1990s)**:
- **Medical diagnosis**: 99% healthy, 1% disease
- **Fraud detection**: 99.9% legitimate, 0.1% fraud
- **Naive approach**: Predict majority class → 99% accuracy but useless!
- **Realization**: Accuracy is misleading for imbalanced data

**Early Solutions (2000s)**:
- **2000**: SMOTE (Synthetic Minority Over-sampling Technique)
- **2002**: Cost-sensitive learning
- **2003**: Ensemble methods (EasyEnsemble, BalanceCascade)
- **2008**: ADASYN (Adaptive Synthetic Sampling)

**Modern Deep Learning Era (2015+)**:
- **2016**: Focal Loss (addressed object detection imbalance)
- **2017**: Class-balanced loss for long-tailed recognition
- **2019**: Deferred re-weighting for imbalanced regression
- **2020**: Recognition that imbalance ≠ always a problem

**Current Understanding**:
- **Not all imbalance is bad**: Natural distributions exist
- **Real problem**: Insufficient minority samples
- **Solutions**: Depends on degree and domain
- **Best practice**: Try multiple approaches, compare

## 🎯 What You'll Learn

1. **Detecting Imbalance** - Metrics and visualization
2. **Random Oversampling** - Duplicate minority samples
3. **Random Undersampling** - Remove majority samples
4. **SMOTE for Text** - Synthetic samples (with limitations)
5. **Class Weighting** - Penalize errors on minority class
6. **Stratified Sampling** - Maintain distribution in splits
7. **Method Comparison** - When to use what
8. **Evaluation Metrics** - Beyond accuracy

## 💡 Why This Matters

**Real-World Examples**:

**Amazon's Hiring AI (2018)**:
- Training data: 10:1 male:female resumes
- Model learned: Male = preferred
- Solution: Project abandoned (too biased)

**Credit Card Fraud Detection**:
- Imbalance: 99.9% legitimate, 0.1% fraud
- Challenge: Missing fraud costs $1000s, false alarm costs $1
- Solution: Cost-sensitive learning + undersampling

**Medical Diagnosis (COVID-19)**:
- Imbalance: 95% negative, 5% positive
- Challenge: False negative = dangerous
- Solution: Oversample positive + class weights

**Lesson**: Ignoring imbalance = Biased, unreliable models

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from typing import List, Tuple, Dict
from pathlib import Path

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# HuggingFace
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Set random seed
np.random.seed(42)

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## Part 1: Understanding and Detecting Class Imbalance

### What is Class Imbalance?

**Definition**: Unequal distribution of classes in a dataset

### Severity Levels:

| Ratio | Severity | Example | Action Needed |
|-------|----------|---------|---------------|
| 1:1 to 1:3 | Mild | 75% vs 25% | Usually okay, maybe stratified sampling |
| 1:3 to 1:10 | Moderate | 90% vs 10% | Consider resampling or class weights |
| 1:10 to 1:100 | Severe | 99% vs 1% | Definitely need intervention |
| >1:100 | Extreme | 99.9% vs 0.1% | Anomaly detection approach might be better |

### Why It's a Problem:

**1. Biased Predictions**
```
Dataset: 95% class A, 5% class B
Model: Always predict A → 95% accuracy!
But: Never predicts B (complete failure)
```

**2. Poor Minority Performance**
- Model doesn't see enough examples
- Can't learn minority class patterns

**3. Misleading Metrics**
- Accuracy looks high but misleading
- Need: F1, Precision, Recall, AUC-ROC

In [ ]:
print("=" * 60)
print("CREATING IMBALANCED DATASET")
print("=" * 60)

# Create a realistic imbalanced text classification dataset
np.random.seed(42)

# Define sample sizes for each class (severe imbalance)
n_majority = 900  # 90%
n_minority = 100  # 10%

# Generate synthetic text samples
def generate_samples(n_samples, label, keywords):
    """Generate synthetic text samples with specific keywords."""
    samples = []
    templates = [
        "This is {} text about {}.",
        "I think {} is very {}.",
        "The {} is {}.",
        "{} and {} are related.",
        "Consider the {} and {}.",
    ]
    
    for i in range(n_samples):
        template = np.random.choice(templates)
        # Use keywords to make classes somewhat separable
        words = np.random.choice(keywords, size=2, replace=True)
        text = template.format(words[0], words[1])
        samples.append({'text': text, 'label': label})
    
    return samples

# Keywords for each class
negative_keywords = ['bad', 'terrible', 'awful', 'poor', 'disappointing', 'hate', 'worst', 'horrible']
positive_keywords = ['good', 'great', 'excellent', 'amazing', 'wonderful', 'love', 'best', 'fantastic']

# Generate samples
negative_samples = generate_samples(n_majority, 'negative', negative_keywords)
positive_samples = generate_samples(n_minority, 'positive', positive_keywords)

# Combine and create dataset
all_samples = negative_samples + positive_samples
np.random.shuffle(all_samples)

texts = [s['text'] for s in all_samples]
labels = [s['label'] for s in all_samples]

# Create dataset
dataset = Dataset.from_dict({
    'text': texts,
    'label': labels
})

print(f"\nDataset created: {len(dataset)} samples")
print(f"Negative: {n_majority} samples ({n_majority/len(dataset)*100:.1f}%)")
print(f"Positive: {n_minority} samples ({n_minority/len(dataset)*100:.1f}%)")
print(f"\nImbalance ratio: {n_majority/n_minority:.1f}:1")

# Calculate imbalance metrics
label_counts = Counter(labels)
majority_class = max(label_counts, key=label_counts.get)
minority_class = min(label_counts, key=label_counts.get)
imbalance_ratio = label_counts[majority_class] / label_counts[minority_class]

print(f"\nMajority class: '{majority_class}' with {label_counts[majority_class]} samples")
print(f"Minority class: '{minority_class}' with {label_counts[minority_class]} samples")
print(f"\nSeverity: ", end="")
if imbalance_ratio < 3:
    print("Mild (ratio < 3)")
elif imbalance_ratio < 10:
    print("Moderate (3 ≤ ratio < 10)")
elif imbalance_ratio < 100:
    print("Severe (10 ≤ ratio < 100)")
else:
    print("Extreme (ratio ≥ 100)")

print("=" * 60)

In [ ]:
# Visualize class distribution
print("=" * 60)
print("CLASS DISTRIBUTION VISUALIZATION")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Bar chart
axes[0].bar(label_counts.keys(), label_counts.values(), 
            color=['coral' if k == minority_class else 'steelblue' for k in label_counts.keys()],
            alpha=0.7)
axes[0].set_title('Class Distribution (Bar Chart)')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(label_counts.items()):
    axes[0].text(i, count + 10, str(count), ha='center', fontweight='bold')

# Plot 2: Pie chart
colors = ['coral' if k == minority_class else 'steelblue' for k in label_counts.keys()]
axes[1].pie(label_counts.values(), labels=label_counts.keys(), autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Class Distribution (Pie Chart)')

# Plot 3: Comparison with balanced
balanced_count = len(dataset) // len(label_counts)
x = np.arange(len(label_counts))
width = 0.35

axes[2].bar(x - width/2, label_counts.values(), width, label='Current', alpha=0.7, color='steelblue')
axes[2].bar(x + width/2, [balanced_count] * len(label_counts), width, 
            label='Balanced', alpha=0.7, color='green')
axes[2].set_title('Current vs Balanced Distribution')
axes[2].set_ylabel('Count')
axes[2].set_xticks(x)
axes[2].set_xticklabels(label_counts.keys())
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Imbalance clearly visible in visualizations")
print("=" * 60)

## Part 2: Random Oversampling

### How It Works:
1. Randomly duplicate minority class samples
2. Repeat until classes are balanced

### Pros:
- ✓ Simple and fast
- ✓ No information loss
- ✓ Works with any classifier

### Cons:
- ✗ Risk of overfitting (exact duplicates)
- ✗ Increases training time
- ✗ Doesn't add new information

### When to Use:
- Small to moderate imbalance (1:10)
- Have enough minority samples (>100)
- Quick baseline approach

In [ ]:
print("=" * 60)
print("RANDOM OVERSAMPLING")
print("=" * 60)

def random_oversample(texts, labels, random_state=42):
    """
    Apply random oversampling to balance classes.
    
    Duplicates minority class samples until all classes have equal counts.
    """
    # Convert to numpy arrays for easier indexing
    texts_array = np.array(texts)
    labels_array = np.array(labels)
    
    # Get class counts
    unique_labels, counts = np.unique(labels_array, return_counts=True)
    max_count = max(counts)
    
    # Oversample each class
    resampled_texts = []
    resampled_labels = []
    
    np.random.seed(random_state)
    
    for label in unique_labels:
        # Get indices for this class
        mask = labels_array == label
        class_texts = texts_array[mask]
        class_labels = labels_array[mask]
        
        current_count = len(class_texts)
        
        if current_count < max_count:
            # Need to oversample
            n_samples_needed = max_count - current_count
            
            # Randomly sample with replacement
            oversample_indices = np.random.choice(
                len(class_texts),
                size=n_samples_needed,
                replace=True
            )
            
            oversampled_texts = class_texts[oversample_indices]
            oversampled_labels = class_labels[oversample_indices]
            
            # Combine original + oversampled
            resampled_texts.extend(class_texts)
            resampled_texts.extend(oversampled_texts)
            resampled_labels.extend(class_labels)
            resampled_labels.extend(oversampled_labels)
        else:
            # Already at max count
            resampled_texts.extend(class_texts)
            resampled_labels.extend(class_labels)
    
    return list(resampled_texts), list(resampled_labels)

# Apply random oversampling
print("\nBefore oversampling:")
print(f"  Total samples: {len(texts)}")
print(f"  Class distribution: {Counter(labels)}")

oversampled_texts, oversampled_labels = random_oversample(texts, labels)

print("\nAfter oversampling:")
print(f"  Total samples: {len(oversampled_texts)}")
print(f"  Class distribution: {Counter(oversampled_labels)}")

# Calculate increase
increase = len(oversampled_texts) - len(texts)
increase_pct = (increase / len(texts)) * 100
print(f"\nDataset size increased by: {increase} samples ({increase_pct:.1f}%)")

# Create oversampled dataset
dataset_oversampled = Dataset.from_dict({
    'text': oversampled_texts,
    'label': oversampled_labels
})

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
before_counts = Counter(labels)
axes[0].bar(before_counts.keys(), before_counts.values(), alpha=0.7, color='steelblue')
axes[0].set_title('Before Oversampling')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(before_counts.items()):
    axes[0].text(i, count + 10, str(count), ha='center', fontweight='bold')

# After
after_counts = Counter(oversampled_labels)
axes[1].bar(after_counts.keys(), after_counts.values(), alpha=0.7, color='green')
axes[1].set_title('After Oversampling')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(after_counts.items()):
    axes[1].text(i, count + 10, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("PROS AND CONS")
print("=" * 60)
print("✓ PROS:")
print("  - Classes are now perfectly balanced")
print("  - Simple and fast")
print("  - No information loss")
print("\n✗ CONS:")
print("  - Exact duplicates → potential overfitting")
print("  - Dataset size increased → more training time")
print("  - No new information added")
print("=" * 60)

## Part 3: Random Undersampling

### How It Works:
1. Randomly remove majority class samples
2. Remove until classes are balanced

### Pros:
- ✓ Fast training (smaller dataset)
- ✓ No overfitting risk
- ✓ Simple to implement

### Cons:
- ✗ Information loss (throw away data)
- ✗ Might lose important patterns
- ✗ Only works if majority class has many samples

### When to Use:
- Very large majority class (100k+ samples)
- Limited training time/resources
- Imbalance is extreme (1:100+)

In [ ]:
print("=" * 60)
print("RANDOM UNDERSAMPLING")
print("=" * 60)

def random_undersample(texts, labels, random_state=42):
    """
    Apply random undersampling to balance classes.
    
    Removes majority class samples until all classes have equal counts.
    """
    texts_array = np.array(texts)
    labels_array = np.array(labels)
    
    # Get class counts
    unique_labels, counts = np.unique(labels_array, return_counts=True)
    min_count = min(counts)
    
    # Undersample each class
    resampled_texts = []
    resampled_labels = []
    
    np.random.seed(random_state)
    
    for label in unique_labels:
        # Get indices for this class
        mask = labels_array == label
        class_texts = texts_array[mask]
        class_labels = labels_array[mask]
        
        current_count = len(class_texts)
        
        if current_count > min_count:
            # Need to undersample
            # Randomly select min_count samples
            undersample_indices = np.random.choice(
                len(class_texts),
                size=min_count,
                replace=False
            )
            
            undersampled_texts = class_texts[undersample_indices]
            undersampled_labels = class_labels[undersample_indices]
            
            resampled_texts.extend(undersampled_texts)
            resampled_labels.extend(undersampled_labels)
        else:
            # Already at min count
            resampled_texts.extend(class_texts)
            resampled_labels.extend(class_labels)
    
    return list(resampled_texts), list(resampled_labels)

# Apply random undersampling
print("\nBefore undersampling:")
print(f"  Total samples: {len(texts)}")
print(f"  Class distribution: {Counter(labels)}")

undersampled_texts, undersampled_labels = random_undersample(texts, labels)

print("\nAfter undersampling:")
print(f"  Total samples: {len(undersampled_texts)}")
print(f"  Class distribution: {Counter(undersampled_labels)}")

# Calculate decrease
decrease = len(texts) - len(undersampled_texts)
decrease_pct = (decrease / len(texts)) * 100
print(f"\nDataset size decreased by: {decrease} samples ({decrease_pct:.1f}%)")
print(f"⚠️  Lost {decrease} samples from majority class!")

# Create undersampled dataset
dataset_undersampled = Dataset.from_dict({
    'text': undersampled_texts,
    'label': undersampled_labels
})

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
before_counts = Counter(labels)
axes[0].bar(before_counts.keys(), before_counts.values(), alpha=0.7, color='steelblue')
axes[0].set_title('Before Undersampling')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(before_counts.items()):
    axes[0].text(i, count + 10, str(count), ha='center', fontweight='bold')

# After
after_counts = Counter(undersampled_labels)
axes[1].bar(after_counts.keys(), after_counts.values(), alpha=0.7, color='coral')
axes[1].set_title('After Undersampling')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(after_counts.items()):
    axes[1].text(i, count + 10, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("PROS AND CONS")
print("=" * 60)
print("✓ PROS:")
print("  - Classes are now balanced")
print("  - Smaller dataset → faster training")
print("  - No overfitting from duplicates")
print("\n✗ CONS:")
print("  - Lost 80% of original data!")
print("  - Might have removed important patterns")
print("  - Model sees less data overall")
print("\n💡 RECOMMENDATION:")
print("  - Only use when majority class is VERY large (10k+ samples)")
print("  - Consider combining with oversampling (hybrid approach)")
print("=" * 60)

## Part 4: Class Weighting (Most Practical for Deep Learning)

### How It Works:
1. Calculate weight for each class: `weight[i] = total_samples / (n_classes * samples_in_class[i])`
2. Apply weights to loss function
3. Minority class errors penalized more

### Pros:
- ✓ No data modification needed
- ✓ No training time increase
- ✓ No information loss
- ✓ **Best for deep learning**

### Cons:
- ✗ Requires loss function support
- ✗ Hyperparameter (weight values)
- ✗ Can cause instability if weights too high

### When to Use:
- **Default choice for deep learning**
- Any imbalance ratio
- When you can't modify data
- PyTorch/TensorFlow/HuggingFace training

In [ ]:
print("=" * 60)
print("CLASS WEIGHTING")
print("=" * 60)

# Calculate class weights
def calculate_class_weights(labels):
    """
    Calculate class weights for imbalanced dataset.
    
    Formula: weight[i] = total_samples / (n_classes * samples_in_class[i])
    """
    labels_array = np.array(labels)
    unique_labels = np.unique(labels_array)
    
    # Calculate weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=unique_labels,
        y=labels_array
    )
    
    # Create dictionary
    weight_dict = {label: weight for label, weight in zip(unique_labels, class_weights)}
    
    return weight_dict, class_weights

# Calculate weights for our dataset
weight_dict, class_weights = calculate_class_weights(labels)

print("\nClass weights:")
print("-" * 40)
for label, weight in weight_dict.items():
    count = labels.count(label)
    print(f"  {label:10s}: weight = {weight:.3f}  (count: {count})")

print("\nInterpretation:")
print("-" * 40)
majority = max(weight_dict, key=lambda k: labels.count(k))
minority = min(weight_dict, key=lambda k: labels.count(k))
print(f"  Minority class '{minority}' has weight {weight_dict[minority]:.3f}")
print(f"  → Errors on this class cost {weight_dict[minority]:.1f}x more")
print(f"\n  Majority class '{majority}' has weight {weight_dict[majority]:.3f}")
print(f"  → Errors on this class cost {weight_dict[majority]:.1f}x more")
print(f"\n  This encourages model to pay more attention to minority class")

# Show how to use in PyTorch
print("\n" + "=" * 60)
print("USAGE IN PYTORCH")
print("=" * 60)
print("""
import torch
import torch.nn as nn

# Convert to tensor
class_weights_tensor = torch.FloatTensor(list(weight_dict.values()))

# Use in loss function
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# During training
loss = criterion(outputs, labels)
""")

# Show how to use in HuggingFace Trainer
print("\n" + "=" * 60)
print("USAGE IN HUGGINGFACE TRAINER")
print("=" * 60)
print("""
from transformers import Trainer, TrainingArguments
import torch

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Apply class weights
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), 
                       labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

# Use the weighted trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
""")

# Visualize weights
plt.figure(figsize=(10, 5))
plt.bar(weight_dict.keys(), weight_dict.values(), alpha=0.7, 
        color=['coral' if k == minority else 'steelblue' for k in weight_dict.keys()])
plt.title('Class Weights')
plt.ylabel('Weight')
plt.grid(axis='y', alpha=0.3)

# Add value labels
for i, (label, weight) in enumerate(weight_dict.items()):
    plt.text(i, weight + 0.05, f"{weight:.3f}", ha='center', fontweight='bold')

plt.show()

print("\n" + "=" * 60)
print("PROS AND CONS")
print("=" * 60)
print("✓ PROS:")
print("  - No data modification (use original dataset)")
print("  - No training time increase")
print("  - No information loss")
print("  - Works well with deep learning")
print("  - Easy to implement in PyTorch/HuggingFace")
print("\n✗ CONS:")
print("  - Requires loss function support")
print("  - Might cause training instability if weights too high")
print("  - Not all frameworks support it easily")
print("\n💡 RECOMMENDATION:")
print("  - **First choice for transformer fine-tuning**")
print("  - Combine with stratified sampling for best results")
print("  - Monitor training loss carefully")
print("=" * 60)

## Part 5: Stratified Sampling (Critical for Evaluation)

### What It Is:
- Maintain class distribution in train/val/test splits
- NOT a resampling method (doesn't change class balance)
- Ensures each split is representative

### Why It Matters:

**Without Stratification**:
```
Original: 90% negative, 10% positive
Random split might give:
  Train: 95% negative, 5% positive
  Test:  80% negative, 20% positive
→ Test distribution doesn't match train!
```

**With Stratification**:
```
Original: 90% negative, 10% positive
Stratified split gives:
  Train: 90% negative, 10% positive
  Test:  90% negative, 10% positive
→ All splits have same distribution
```

### When to Use:
- **Always** for imbalanced classification
- Especially critical with small datasets
- Ensures reliable evaluation metrics

In [ ]:
print("=" * 60)
print("STRATIFIED SAMPLING")
print("=" * 60)

# Compare random vs stratified split
print("\nOriginal dataset distribution:")
original_dist = Counter(labels)
for label, count in original_dist.items():
    pct = (count / len(labels)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

# Random split (BAD for imbalanced data)
print("\n" + "=" * 60)
print("RANDOM SPLIT (Not Recommended)")
print("=" * 60)

train_texts_r, test_texts_r, train_labels_r, test_labels_r = train_test_split(
    texts, labels, test_size=0.2, random_state=42, shuffle=True
)

print("\nTrain set distribution:")
train_dist_r = Counter(train_labels_r)
for label, count in train_dist_r.items():
    pct = (count / len(train_labels_r)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

print("\nTest set distribution:")
test_dist_r = Counter(test_labels_r)
for label, count in test_dist_r.items():
    pct = (count / len(test_labels_r)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

# Stratified split (GOOD for imbalanced data)
print("\n" + "=" * 60)
print("STRATIFIED SPLIT (Recommended)")
print("=" * 60)

train_texts_s, test_texts_s, train_labels_s, test_labels_s = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print("\nTrain set distribution:")
train_dist_s = Counter(train_labels_s)
for label, count in train_dist_s.items():
    pct = (count / len(train_labels_s)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

print("\nTest set distribution:")
test_dist_s = Counter(test_labels_s)
for label, count in test_dist_s.items():
    pct = (count / len(test_labels_s)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Original
axes[0, 0].bar(original_dist.keys(), original_dist.values(), alpha=0.7, color='gray')
axes[0, 0].set_title('Original Dataset')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(axis='y', alpha=0.3)

# Random split - train
axes[0, 1].bar(train_dist_r.keys(), train_dist_r.values(), alpha=0.7, color='steelblue')
axes[0, 1].set_title('Random Split - Train')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(axis='y', alpha=0.3)

# Random split - test
axes[0, 2].bar(test_dist_r.keys(), test_dist_r.values(), alpha=0.7, color='coral')
axes[0, 2].set_title('Random Split - Test')
axes[0, 2].set_ylabel('Count')
axes[0, 2].grid(axis='y', alpha=0.3)

# Original (repeated)
axes[1, 0].bar(original_dist.keys(), original_dist.values(), alpha=0.7, color='gray')
axes[1, 0].set_title('Original Dataset')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(axis='y', alpha=0.3)

# Stratified split - train
axes[1, 1].bar(train_dist_s.keys(), train_dist_s.values(), alpha=0.7, color='steelblue')
axes[1, 1].set_title('Stratified Split - Train')
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(axis='y', alpha=0.3)

# Stratified split - test
axes[1, 2].bar(test_dist_s.keys(), test_dist_s.values(), alpha=0.7, color='coral')
axes[1, 2].set_title('Stratified Split - Test')
axes[1, 2].set_ylabel('Count')
axes[1, 2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("KEY OBSERVATION")
print("=" * 60)
print("With stratified split:")
print("  ✓ Train and test have same class distribution as original")
print("  ✓ Evaluation metrics are reliable")
print("  ✓ Model evaluation reflects real performance")
print("\nWith random split:")
print("  ✗ Test might have different distribution")
print("  ✗ Metrics might be misleading")
print("  ✗ Less reliable evaluation")
print("\n💡 ALWAYS use stratified split for imbalanced classification!")
print("=" * 60)

## Part 6: Method Comparison - When to Use What

### Decision Tree:

```
START: Imbalanced classification problem
│
├─ Imbalance < 1:3
│  └─> Just use stratified sampling
│
├─ Imbalance 1:3 to 1:10 (Moderate)
│  ├─ Deep learning?
│  │  └─> YES: Class weighting + stratified sampling
│  └─> NO: Random oversampling or SMOTE
│
├─ Imbalance 1:10 to 1:100 (Severe)
│  ├─ Large majority class (>10k samples)?
│  │  ├─> YES: Undersampling + oversampling (hybrid)
│  │  └─> NO: Oversampling + class weighting
│  └─ Deep learning?
│     └─> YES: Definitely use class weighting
│
└─ Imbalance > 1:100 (Extreme)
   ├─> Consider: Is this really classification?
   │   Maybe anomaly detection is better?
   └─> If classification: Combination approach
       ├─ Undersample majority to 1:10
       ├─ Oversample minority to 1:5
       └─ Apply class weighting
```

In [ ]:
print("=" * 60)
print("METHOD COMPARISON SUMMARY")
print("=" * 60)

comparison_data = {
    'Method': [
        'Random Oversampling',
        'Random Undersampling',
        'Class Weighting',
        'Stratified Sampling',
        'Hybrid (Over+Under)',
    ],
    'Pros': [
        'Simple, no info loss, balanced classes',
        'Fast training, smaller dataset',
        'No data modification, works with DL',
        'Representative splits, reliable eval',
        'Best of both worlds',
    ],
    'Cons': [
        'Overfitting risk, longer training',
        'Information loss, less data',
        'Requires framework support',
        'Doesn\'t fix imbalance, just splits',
        'More complex, hyperparameters',
    ],
    'Best For': [
        'Moderate imbalance (1:10)',
        'Huge majority class (>10k)',
        'Deep learning, any imbalance',
        'ALWAYS use for imbalanced data',
        'Severe imbalance (1:100)',
    ],
    'Training Time': [
        'Longer (more data)',
        'Shorter (less data)',
        'Same as original',
        'Same as original',
        'Medium',
    ],
}

df_comparison = pd.DataFrame(comparison_data)

print("\n", df_comparison.to_string(index=False))

print("\n\n" + "=" * 60)
print("RECOMMENDATIONS BY SCENARIO")
print("=" * 60)

scenarios = [
    {
        'scenario': 'Sentiment Analysis (80:20 negative:positive)',
        'imbalance': 'Moderate (1:4)',
        'recommendation': 'Class weighting + stratified sampling',
        'reason': 'Standard case, DL works well with weighting'
    },
    {
        'scenario': 'Fraud Detection (99.9:0.1 legit:fraud)',
        'imbalance': 'Extreme (1:1000)',
        'recommendation': 'Consider anomaly detection instead',
        'reason': 'Too extreme for classification, treat fraud as anomaly'
    },
    {
        'scenario': 'Medical Diagnosis (95:5 healthy:disease)',
        'imbalance': 'Severe (1:19)',
        'recommendation': 'Oversampling + class weighting + stratified',
        'reason': 'Can\'t afford false negatives, need to boost minority'
    },
    {
        'scenario': 'Spam Detection (90:10 ham:spam)',
        'imbalance': 'Moderate (1:9)',
        'recommendation': 'Class weighting or undersampling',
        'reason': 'Large dataset, undersampling is viable'
    },
]

for i, scenario in enumerate(scenarios, 1):
    print(f"\n{i}. {scenario['scenario']}")
    print(f"   Imbalance: {scenario['imbalance']}")
    print(f"   Recommendation: {scenario['recommendation']}")
    print(f"   Reason: {scenario['reason']}")

print("\n\n" + "=" * 60)
print("GENERAL GUIDELINES")
print("=" * 60)
print("""
1. ALWAYS use stratified sampling (for splits)
   → Ensures reliable evaluation

2. For transformer fine-tuning:
   → Start with class weighting
   → Try oversampling if still underperforming

3. For traditional ML:
   → Try oversampling or SMOTE first
   → Undersampling if dataset is huge

4. Extreme imbalance (>1:100):
   → Question if classification is right approach
   → Consider anomaly detection
   → Or use combination of methods

5. Don't forget:
   → Monitor minority class metrics (precision, recall, F1)
   → Use confusion matrix for evaluation
   → Accuracy is misleading!
""")
print("=" * 60)

## 📊 Summary and Key Takeaways

### What We Learned:

**1. Understanding Imbalance**
- Severity levels: Mild (1:3), Moderate (1:10), Severe (1:100), Extreme (>1:100)
- Problem: Biased predictions, poor minority performance
- Detection: Class distribution visualization, imbalance ratio

**2. Random Oversampling**
- Duplicate minority samples
- Pros: Simple, no information loss
- Cons: Overfitting risk, longer training
- Use: Moderate imbalance, enough minority samples

**3. Random Undersampling**
- Remove majority samples
- Pros: Faster training, no overfitting
- Cons: Information loss, less data
- Use: Very large majority class only

**4. Class Weighting** ⭐
- Weight loss by class frequency
- Pros: No data modification, works with DL
- Cons: Requires framework support
- **Use: Default choice for deep learning**

**5. Stratified Sampling** ⭐
- Maintain class distribution in splits
- Pros: Reliable evaluation
- Cons: Doesn't fix imbalance
- **Use: ALWAYS for imbalanced data**

### Quick Reference Guide:

| Imbalance | Dataset Size | Training Method | Recommendation |
|-----------|--------------|-----------------|----------------|
| <1:3 | Any | Any | Stratified sampling only |
| 1:3 to 1:10 | Any | Deep Learning | Class weighting + stratified |
| 1:3 to 1:10 | Any | Traditional ML | Oversampling + stratified |
| 1:10 to 1:100 | Small (<10k) | Any | Oversample + class weight |
| 1:10 to 1:100 | Large (>10k) | Any | Hybrid (over+under) + weight |
| >1:100 | Any | Any | Consider anomaly detection |

### Evaluation Metrics for Imbalanced Data:

**DON'T use:**
- ✗ Accuracy (misleading)

**DO use:**
- ✓ Precision (positive predictions that are correct)
- ✓ Recall (actual positives that are found)
- ✓ F1-score (harmonic mean of precision & recall)
- ✓ AUC-ROC (area under ROC curve)
- ✓ Confusion matrix (visualize all errors)

### Implementation Checklist:

**Before Training:**
```python
# 1. Analyze imbalance
✓ Visualize class distribution
✓ Calculate imbalance ratio
✓ Determine severity

# 2. Choose methods
✓ Stratified sampling (ALWAYS)
✓ Class weighting (for DL)
✓ Resampling (if needed)

# 3. Implement
✓ Split data (stratified)
✓ Calculate class weights
✓ Apply to loss function

# 4. Evaluate
✓ Use proper metrics (F1, AUC-ROC)
✓ Check confusion matrix
✓ Monitor minority class performance
```

### Common Mistakes to Avoid:

| Mistake | Why It's Bad | Solution |
|---------|--------------|----------|
| Random split | Test distribution differs | Use stratified split |
| Only accuracy | Misleading metric | Use F1, precision, recall |
| No weighting | Model ignores minority | Apply class weights |
| Oversample before split | Data leakage | Split first, then oversample |
| Undersample small dataset | Lose critical info | Use oversampling instead |

### Next Notebook:
**06_data_augmentation_nlp.ipynb**

We'll explore:
- Back-translation for paraphrasing
- Synonym replacement with WordNet
- Contextual replacement with BERT
- Random insertion, deletion, swap
- Paraphrasing with LLMs (GPT-4 API)
- Noise injection techniques
- Synthetic data generation
- Quality filtering augmented data
- Comparing augmentation methods

---

**Historical Note**: The class imbalance problem was identified in the 1990s with medical diagnosis systems, but wasn't widely recognized in ML until the 2000s. SMOTE (2000) was revolutionary, showing that synthetic data generation could outperform simple resampling. The deep learning era brought new challenges - early studies in 2015-2016 showed that CNNs struggled with imbalanced data more than traditional ML. This led to the development of focal loss (2017) and class-balanced loss (2017), specifically designed for deep learning. Today's best practice: **class weighting is the first choice for transformer fine-tuning**, with resampling as a supplement for severe cases.